<a href="https://colab.research.google.com/github/Chisman001/ML-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chisman001/ML-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.97 MiB/s, done.
Resolving deltas: 100% (153/153), done.


In [19]:
%cd flyrank-ml-internship-starter

/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [20]:
# Install required packages
!pip -q install duckdb huggingface_hub pyarrow

import duckdb
from google.colab import userdata

# Get your Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

In [21]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("Successfully connected to Hugging Face!")

Successfully connected to Hugging Face!


In [22]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git /content/flyrank-ml-internship-starter

fatal: destination path '/content/flyrank-ml-internship-starter' already exists and is not an empty directory.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page for one client on one report date (client × content × date). I will use March 2026 as the mid-panel development window, which contains 9,841,378 rows covering 2026-03-01 to 2026-03-31. I will not use the final June 2026 month for development because it is treated as a sealed test window. Client data availability may also differ across the panel, so usable history will be checked rather than assumed.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", HF_TOKEN is not None)


HF_TOKEN available: True


In [24]:
!pip -q install duckdb huggingface_hub

In [25]:
import duckdb

con = duckdb.connect()

print("DuckDB connected.")

DuckDB connected.


In [26]:
from huggingface_hub import login

login(token=HF_TOKEN, add_to_git_credential=False)

print("Hugging Face authentication successful.")

Hugging Face authentication successful.


In [27]:
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Hugging Face authentication configured for DuckDB.")

Hugging Face authentication configured for DuckDB.


In [28]:
import os

march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The planned features are:

- `gsc_impressions` — search visibility available at the prediction point.
- `gsc_clicks` — search clicks available at the prediction point.
- `gsc_avg_position` — search ranking performance available at the prediction point.
- `sessions_organic` — organic traffic available at the prediction point.
- `word_count` — content-length context, provided the value is available before the prediction point.

### Label / proxy

The label will represent whether a content page becomes a refresh opportunity based on its subsequent performance. The outcome must come from a future window after the feature window so that the model does not use future information when making the prediction.

### Context

- `client_hash_id` — used for client-level grouping, joining, and splitting.
- `content_hash_id` — identifies the content item.
- `report_date` — defines the observation date and preserves chronological order.
- `month` — used to select the warehouse partition and development window.

### Excluded

- `gsc_data_available` — used to determine whether GSC data exists, not as a content-performance feature.
- `ga4_data_available` — used to determine whether GA4 data exists, not as a content-performance feature.
- `client_has_gsc` and `client_has_ga4` — describe data availability rather than content performance.
- Any future performance or outcome-derived field — excluded to prevent target leakage.
- `is_deleted` — excluded because deletion status can represent a downstream business decision rather than a signal available when deciding whether a page needs a refresh.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
    SELECT *
    FROM read_parquet('{march_path}')
    LIMIT 5
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [30]:
con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
    LIMIT 5
""").show()

┌─────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────┬──────────────────────┬──────────────────────┬─────────────────┬───────────────┬─────────────┬───────────────────┬────────┬───────────────┬───────────┬────────────────┬──────────────────────┬─────────────────────────┬────────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────┬──────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │     keyword_hash_id      │     url_hash_id      │ keyword_char_count │ keyword_token_count │ url_char_count │ content_created_date │ content_updated_date │  content_type   │ search_volume │ competition │ competition_level │  cpc   │  main_intent  │ backlinks │ category_count │ keyword_created_date │      provider_used      │       model_used       │ char_count │ word_count │ last_optimized_date │ optimization_eligible_date │ is_p

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1
The grain check returned 0 rows, confirming that each row represents one client-content combination for one report date in the March 2026 partition.

### Query 2
The March 2026 partition contains 9,841,378 rows and covers the period from 2026-03-01 to 2026-03-31.

### Query 3
GSC data is available for 3,611,061 of the 9,841,378 March rows. The remaining 6,230,317 rows do not have usable GSC data. This confirms that GSC availability is not universal and should be explicitly checked rather than treating missing search data as zero performance.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 1: Verify the grain
con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

# Query 2: Verify row count and date window
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").show()

# Query 3: Verify GSC data availability
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_unavailable_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬──────────────────────┐
│ total_rows │ gsc_available_rows │ gsc_unavailable_rows │
│   int64    │       int64        │        int64         │
├────────────┼────────────────────┼──────────────────────┤
│    9841378 │            3611061 │              6230317 │
└────────────┴────────────────────┴──────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This data can show observed search and traffic performance, but it cannot fully explain why a page is performing the way it is.

Client history is unbalanced, so different clients have different amounts of available search and analytics history. Some early rows may have GSC data without GA4 data, so unavailable analytics data should not be interpreted as zero engagement.

The data also cannot directly tell us whether a content change caused a performance change. It can identify pages with signals that suggest a refresh may be useful, but the model should be treated as decision support rather than proof that a refresh will improve performance.

The query table also uses a fixed 90-day window that can overlap with performance periods, so features must be aligned carefully with the prediction window to avoid leakage.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.